# Workflow: aggiunta dataset `cluster` con nuova maschera

**Struttura attesa dei file H5 di input:**
```
V1/
  Metadata/  (redshift, ra, dec, photo_catalog, units_*)
  Photometry/ (flux, flux_err, filters)
  Spectroscopy/ (wavelength, flux, flux_err, mask)  ← mask da sostituire
V2/  (opzionale, se ci sono più cataloghi fotometrici)
  ...
```

**Output:** aggiunta del gruppo `cluster/` al file esistente, identico a `V1` ma con la nuova maschera.

---
## Step 1 — Crea e salva la nuova maschera in un file H5 temporaneo

In [1]:
import h5py
import numpy as np
import os


# def save_mask_to_h5(mask: np.ndarray, mask_file: str) -> None:
#     """
#     Step 1: salva la nuova maschera in un file H5 temporaneo.

#     Parameters
#     ----------
#     mask      : array booleano (True = pixel valido, False = mascherato),
#                 deve avere la stessa lunghezza dello spettro.
#     mask_file : path del file H5 temporaneo di output.
#     """
#     os.makedirs(os.path.dirname(mask_file) if os.path.dirname(mask_file) else ".", exist_ok=True)
#     mask = np.asarray(mask, dtype=bool)
#     with h5py.File(mask_file, "w") as f:
#         f.create_dataset("mask", data=mask)
#     print(f"✅ Maschera salvata in: {mask_file}  |  shape={mask.shape}  |  True={mask.sum()}  False={(~mask).sum()}")


# ── Esempio d'uso ──────────────────────────────────────────────────────────────
# Qui costruisci/importa la tua maschera. Esempio fittizio:
# new_mask = my_clustering_function(...)   # True = da fittare, False = da escludere

# new_mask = np.ones(1000, dtype=bool)   # placeholder
# save_mask_to_h5(new_mask, "/path/to/temp_mask.h5")

---
## Step 2 — Aggiungi il dataset `cluster` al file H5 originale

In [2]:
def add_cluster_version(
    data_file: str,
    mask_file: str,
    source_version: str = "V1",
    cluster_name: str = "cluster",
    overwrite: bool = False,
) -> None:
    """
    Step 2: apre `data_file` in modalità append e aggiunge il gruppo
    `cluster_name` copiando tutti i dati da `source_version` ma
    sostituendo `Spectroscopy/mask` con quella letta da `mask_file`.

    Parameters
    ----------
    data_file      : file H5 da modificare (viene aperto in append 'a').
    mask_file      : file H5 temporaneo che contiene il dataset 'mask'.
    source_version : gruppo da cui copiare i dati (default 'V1').
    cluster_name   : nome del nuovo gruppo da creare (default 'cluster').
    overwrite      : se True, elimina il gruppo esistente prima di riscriverlo.
    """
    # --- Leggiamo la nuova maschera ---
    with h5py.File(mask_file, "r") as f_mask:
        new_mask = np.asarray(f_mask["mask"][()], dtype=bool)

    with h5py.File(data_file, "a") as f:

        # Controllo versione sorgente
        if source_version not in f:
            raise KeyError(f"Il gruppo '{source_version}' non esiste in {data_file}. "
                           f"Gruppi disponibili: {list(f.keys())}")

        src = f[source_version]

        # Controllo coerenza lunghezza maschera
        if "Spectroscopy/wavelength" in src:
            n_wave = src["Spectroscopy/wavelength"].shape[0]
            if new_mask.shape[0] != n_wave:
                raise ValueError(
                    f"Lunghezza maschera ({new_mask.shape[0]}) diversa dalla griglia "
                    f"spettrale ({n_wave}). Controlla il file maschera."
                )

        # Gestione overwrite
        if cluster_name in f:
            if overwrite:
                del f[cluster_name]
                print(f"⚠️  Gruppo '{cluster_name}' esistente eliminato (overwrite=True).")
            else:
                raise ValueError(
                    f"Il gruppo '{cluster_name}' esiste già in {data_file}. "
                    "Usa overwrite=True per sovrascriverlo."
                )

        # --- Creiamo il nuovo gruppo ---
        new_grp = f.create_group(cluster_name)

        # 1. Metadata — copia tutto 1:1
        if "Metadata" in src:
            meta_src = src["Metadata"]
            meta_dst = new_grp.create_group("Metadata")
            for key in meta_src.keys():
                meta_dst.create_dataset(key, data=meta_src[key][()])

        # 2. Photometry — copia tutto 1:1
        if "Photometry" in src:
            phot_src = src["Photometry"]
            phot_dst = new_grp.create_group("Photometry")
            for key in phot_src.keys():
                phot_dst.create_dataset(key, data=phot_src[key][()])

        # 3. Spectroscopy — copia tutto ma sostituisce mask
        if "Spectroscopy" in src:
            spec_src = src["Spectroscopy"]
            spec_dst = new_grp.create_group("Spectroscopy")
            for key in spec_src.keys():
                if key == "mask":
                    spec_dst.create_dataset("mask", data=new_mask)  # ← NUOVA maschera
                else:
                    spec_dst.create_dataset(key, data=spec_src[key][()])

    print(f"✅ Gruppo '{cluster_name}' aggiunto a: {data_file}")
    print(f"   Maschera: True={new_mask.sum()}  False={(~new_mask).sum()}  shape={new_mask.shape}")


# ── Esempio d'uso su un singolo file ──────────────────────────────────────────
# add_cluster_version(
#     data_file="/path/to/source.h5",
#     mask_file="/path/to/temp_mask.h5",
#     source_version="V1",
#     cluster_name="cluster",
# )

---
## Step 3 — Workflow completo su tutti i file del campione

In [4]:
def full_cluster_workflow(
    data_file: str,
    mask_dir: str,
    source_version: str = "V1",
    cluster_name: str = "cluster",
    overwrite: bool = False,
) -> None:
    """
    Workflow completo per un singolo file:
      1) Calcola la nuova maschera leggendo lo spettro da data_file
      2) La salva in un file H5 temporaneo in mask_dir
      3) Aggiunge il dataset cluster al file originale
      4) (Opzionale) rimuove il file temporaneo

    Modifica `compute_new_mask` con la tua logica.
    """
    source_name = os.path.splitext(os.path.basename(data_file))[0]
    mask_file = os.path.join(mask_dir, f"{source_name}.h5")

    # ── 1. Leggi lo spettro e calcola la nuova maschera ──
    with h5py.File(data_file, "r") as f:
        wave = f[f"{source_version}/Spectroscopy/wavelength"][()]
        flux = f[f"{source_version}/Spectroscopy/flux"][()]
        flux_err = f[f"{source_version}/Spectroscopy/flux_err"][()]
        old_mask = f[f"{source_version}/Spectroscopy/mask"][()]

    # ── Inserisci qui la tua logica di clustering / selezione ──────────────
    # ────────────────────────────────────────────────────────────────────────

    # ── 2. Salva la maschera temporanea ──
    # save_mask_to_h5(new_mask, mask_file)

    # ── 3. Aggiungi il dataset cluster ──
    add_cluster_version(data_file, mask_file, source_version, cluster_name, overwrite)

    # ── 4. Rimuovi il file temporaneo (opzionale) ──
    # os.remove(mask_file)
    print(f"   Temp mask eliminata: {mask_file}\n")


# ── Esecuzione su tutti i file del campione ────────────────────────────────────
# data_dir = "/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/DATA"
# mask_dir = "/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/MASKS_TEMP"


In [20]:
# Lista file (adatta al tuo caso)
import glob

data_files =["/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/capers-egs61-v4_prism-clear_6368_24713.h5"]# sorted(glob.glob("/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/*.h5"))

print(f"File trovati: {len(data_files)}\n")

for data_file in data_files:
    try:
        full_cluster_workflow(
            data_file=data_file,
            mask_dir="/Users/matteo/EasyProspector/tmp",
            source_version="V1",
            cluster_name="cluster",
            overwrite=True,
        )
    except Exception as e:
        print(f"❌ Errore su {os.path.basename(data_file)}: {e}\n")


File trovati: 1

⚠️  Gruppo 'cluster' esistente eliminato (overwrite=True).
✅ Gruppo 'cluster' aggiunto a: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/capers-egs61-v4_prism-clear_6368_24713.h5
   Maschera: True=254  False=219  shape=(473,)
   Temp mask eliminata: /Users/matteo/EasyProspector/tmp/capers-egs61-v4_prism-clear_6368_24713.h5



---
## Step 4 — Verifica il risultato su un file campione

In [6]:
def inspect_h5(filepath: str) -> None:
    """Stampa la struttura completa di un file H5 con shape dei dataset."""
    def _print_tree(name, obj):
        indent = "  " * name.count("/")
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}📄 {name.split('/')[-1]}  shape={obj.shape}  dtype={obj.dtype}")
        else:
            print(f"{indent}📁 {name.split('/')[-1]}/")

    print(f"\n{'='*60}")
    print(f"File: {filepath}")
    print(f"{'='*60}")
    with h5py.File(filepath, "r") as f:
        print(f"Gruppi top-level: {list(f.keys())}\n")
        f.visititems(_print_tree)


# Controlla un file
for data_file in data_files:
    try:
        inspect_h5(data_file)
    except Exception as e:
        print(f"❌ Errore su {os.path.basename(data_file)}: {e}\n")


File: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/capers-egs61-v4_prism-clear_6368_24713.h5
Gruppi top-level: ['V1', 'cluster']

📁 V1/
  📁 Metadata/
    📄 dec  shape=()  dtype=float64
    📄 photo_catalog  shape=()  dtype=object
    📄 ra  shape=()  dtype=float64
    📄 redshift  shape=()  dtype=float64
    📄 units_photo  shape=()  dtype=object
    📄 units_spec  shape=()  dtype=object
  📁 Photometry/
    📄 filters  shape=(16,)  dtype=object
    📄 flux  shape=(16,)  dtype=float64
    📄 flux_err  shape=(16,)  dtype=float64
  📁 Spectroscopy/
    📄 flux  shape=(473,)  dtype=float64
    📄 flux_err  shape=(473,)  dtype=float64
    📄 mask  shape=(473,)  dtype=bool
    📄 wavelength  shape=(473,)  dtype=float64
📁 cluster/
  📁 Metadata/
    📄 dec  shape=()  dtype=float64
    📄 photo_catalog  shape=()  dtype=object
    📄 ra  shape=()  dtype=float64
    📄 redshift  shape=()  dtype=float64
    📄 units_photo  shape=()  dtype=object
    📄 units_spec  shape=()  dtype=object
  📁

In [15]:
"""
plot_h5.py
──────────
Funzioni di visualizzazione per i file H5 in formato GalaxyDataManager.

Struttura attesa:
  <version>/
    Metadata/   (redshift, ra, dec, ...)
    Photometry/ (flux, flux_err, filters)
    Spectroscopy/ (wavelength, flux, flux_err, mask)
"""

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from typing import Optional
import warnings

# ─────────────────────────────────────────────────────────────────────────────
# STILE GLOBALE
# ─────────────────────────────────────────────────────────────────────────────
STYLE = {
    "figure.facecolor": "#0f1117",
    "axes.facecolor": "#161b27",
    "axes.edgecolor": "#2e3650",
    "axes.labelcolor": "#c8d0e8",
    "axes.titlecolor": "#e8ecf8",
    "axes.grid": True,
    "grid.color": "#2e3650",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.6,
    "xtick.color": "#7a8ab0",
    "ytick.color": "#7a8ab0",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "font.family": "monospace",
    "text.color": "#c8d0e8",
    "lines.antialiased": True,
    "patch.antialiased": True,
}

# Palette colori
C_FLUX = "#5b9ef5"  # spettro flux
C_ERR = "#2d5fa8"  # envelope errore
C_MASK = "#ffffff"  # band maschera
C_PHOT_FLUX = "#f07b3a"  # fotometria flux
C_PHOT_ERR = "#a34d1a"  # fotometria errori
C_ZLINE = "#e8c53a"  # linea redshift / annotazioni
ALPHA_MASK = 0.12  # trasparenza band mascherate


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────


def _load_version(filepath: str, version: str) -> dict:
    """Legge tutti i dataset di una versione e li restituisce come dict."""
    data = {"version": version, "filepath": filepath}
    with h5py.File(filepath, "r") as f:
        if version not in f:
            available = list(f.keys())
            raise KeyError(
                f"Versione '{version}' non trovata. Disponibili: {available}"
            )
        grp = f[version]

        # Metadata
        if "Metadata" in grp:
            for k in grp["Metadata"].keys():
                val = grp["Metadata"][k][()]
                data[k] = val.decode() if isinstance(val, bytes) else val

        # Spectroscopy
        if "Spectroscopy" in grp:
            spec = grp["Spectroscopy"]
            data["wave"] = spec["wavelength"][()] if "wavelength" in spec else None
            data["flux"] = spec["flux"][()] if "flux" in spec else None
            data["flux_err"] = spec["flux_err"][()] if "flux_err" in spec else None
            data["mask"] = (
                np.asarray(spec["mask"][()], dtype=bool) if "mask" in spec else None
            )

        # Photometry
        if "Photometry" in grp:
            phot = grp["Photometry"]
            data["phot_flux"] = phot["flux"][()] if "flux" in phot else None
            data["phot_err"] = phot["flux_err"][()] if "flux_err" in phot else None
            data["phot_filters"] = phot["filters"][()] if "filters" in phot else None

    return data


def _robust_ylim(
    y: np.ndarray,
    mask: Optional[np.ndarray] = None,
    lo_pct: float = 1.0,
    hi_pct: float = 99.0,
    margin: float = 0.12,
) -> tuple[float, float]:
    """
    Calcola limiti Y robusti usando percentili sui pixel non mascherati.
    Aggiunge un margine relativo sopra e sotto.
    """
    valid = y.copy()
    if mask is not None:
        valid = valid[mask]
    valid = valid[np.isfinite(valid)]
    if len(valid) == 0:
        return (0.0, 1.0)
    lo = np.percentile(valid, lo_pct)
    hi = np.percentile(valid, hi_pct)
    span = hi - lo
    pad = span * margin
    return (lo - pad, hi + pad)


def _robust_xlim(
    wave: np.ndarray, mask: Optional[np.ndarray] = None, margin: float = 0.02
) -> tuple[float, float]:
    """Limiti X: ignora NaN, aggiunge piccolo margine."""
    w = wave[np.isfinite(wave)]
    if len(w) == 0:
        return (wave.min(), wave.max())
    lo, hi = w.min(), w.max()
    pad = (hi - lo) * margin
    return (lo - pad, hi + pad)


def _draw_mask_bands(ax, wave: np.ndarray, mask: np.ndarray) -> None:
    """
    Disegna bande grigie verticali per le regioni mascherate (mask == False).
    Regioni contigue vengono unite in un'unica banda.
    """
    if mask is None or wave is None:
        return

    masked = ~mask  # True dove è mascherato
    if not masked.any():
        return

    # Trova i blocchi contigui di pixel mascherati
    diff = np.diff(masked.astype(int), prepend=0, append=0)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]

    dw = np.diff(wave)
    hw = np.median(dw) / 2.0  # mezzo passo per estendere leggermente

    for s, e in zip(starts, ends):
        x0 = wave[s] - hw
        x1 = wave[e - 1] + hw
        ax.axvspan(x0, x1, color=C_MASK, alpha=ALPHA_MASK, linewidth=0, zorder=1)

    # Linea di legenda unica
    ax.axvspan(
        0, 0, color=C_MASK, alpha=0.4, linewidth=0, label="masked region", zorder=1
    )


def _title_from_data(data: dict, version: str) -> str:
    """Costruisce un titolo compatto dal filepath e dai metadati."""
    import os

    name = os.path.splitext(os.path.basename(data["filepath"]))[0]
    z = data.get("redshift", None)
    z_str = f"   z = {z:.4f}" if z is not None and np.isfinite(float(z)) else ""
    return f"{name}   [{version}]{z_str}"


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 — SPETTRO
# ─────────────────────────────────────────────────────────────────────────────


def plot_spectrum(
    filepath: str,
    version: str = "V1",
    ax: Optional[plt.Axes] = None,
    show: bool = True,
    figsize: tuple = (14, 4),
    rest_frame: bool = False,
    highlight_mask: bool = True,
) -> plt.Figure:
    """
    Plotta lo spettro (flux ± err) evidenziando le regioni mascherate.

    Parameters
    ----------
    filepath      : path del file H5
    version       : gruppo da visualizzare (es. 'V1', 'cluster')
    ax            : se fornito, usa quest'asse invece di crearne uno nuovo
    show          : chiama plt.show() alla fine
    figsize       : dimensione figura (ignorata se ax è fornito)
    rest_frame    : se True, mostra lambda rest = lambda_obs / (1+z)
    highlight_mask: se True, disegna le bande grigie sulle regioni mascherate
    """
    data = _load_version(filepath, version)

    wave = data.get("wave")
    flux = data.get("flux")
    flux_err = data.get("flux_err")
    mask = data.get("mask")
    z = data.get("redshift", 0.0)

    if wave is None or flux is None:
        raise ValueError("Spectroscopy/wavelength o flux mancanti nel file.")

    with plt.rc_context(STYLE):
        standalone = ax is None
        if standalone:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.get_figure()

        # Eventuale conversione rest-frame
        wave_plot = wave / (1.0 + float(z)) if rest_frame else wave
        xlabel = (
            r"$\lambda_\mathrm{rest}$ [µm]"
            if rest_frame
            else r"$\lambda_\mathrm{obs}$ [µm]"
        )

        # ── Envelope errore ──────────────────────────────────────────────────
        if flux_err is not None:
            ax.fill_between(
                wave_plot,
                flux - flux_err,
                flux + flux_err,
                color=C_ERR,
                alpha=0.35,
                linewidth=0,
                zorder=2,
                label="flux err",
            )

        # ── Curva flux ───────────────────────────────────────────────────────
        ax.plot(
            wave_plot,
            flux,
            color=C_FLUX,
            linewidth=0.9,
            alpha=0.92,
            zorder=3,
            label="flux",
        )

        # ── Bande mascherate ────────────────────────────────────────────────
        if highlight_mask and mask is not None:
            _draw_mask_bands(ax, wave_plot, mask)

        # ── Limiti robusti ───────────────────────────────────────────────────
        ax.set_xlim(_robust_xlim(wave_plot))
        ax.set_ylim(_robust_ylim(flux, mask))

        # ── Decorazioni ─────────────────────────────────────────────────────
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Flux [maggies]")
        ax.set_title(_title_from_data(data, version), pad=10)

        legend = ax.legend(
            framealpha=0.15,
            edgecolor="#2e3650",
            labelcolor="#c8d0e8",
            fontsize=8,
            loc="upper right",
        )

        # Statistiche sulla maschera nel corner
        if mask is not None:
            n_good = mask.sum()
            n_total = len(mask)
            pct = 100 * n_good / n_total
            ax.text(
                0.01,
                0.97,
                f"coverage: {n_good}/{n_total}  ({pct:.1f}%)",
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=8,
                color="#7a8ab0",
            )

        if standalone:
            fig.tight_layout()
            if show:
                plt.show()

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 — FOTOMETRIA (SED)
# ─────────────────────────────────────────────────────────────────────────────


def plot_photometry(
    filepath: str,
    version: str = "V1",
    ax: Optional[plt.Axes] = None,
    show: bool = True,
    figsize: tuple = (9, 4),
) -> plt.Figure:
    """
    Plotta la SED fotometrica (flux ± err) come scatter con errorbars.
    """
    data = _load_version(filepath, version)

    phot_flux = data.get("phot_flux")
    phot_err = data.get("phot_err")
    filters = data.get("phot_filters")

    if phot_flux is None:
        raise ValueError("Photometry/flux mancante nel file.")

    # Costruisci asse x: usa indice se i filtri non sono wavelength numeriche
    n = len(phot_flux)
    x = np.arange(n)

    filter_labels = []
    if filters is not None:
        for f in filters:
            if isinstance(f, (bytes, np.bytes_)):
                filter_labels.append(f.decode())
            else:
                filter_labels.append(str(f))

    with plt.rc_context(STYLE):
        standalone = ax is None
        if standalone:
            fig, ax = plt.subplots(figsize=figsize)
        else:
            fig = ax.get_figure()

        if phot_err is not None:
            # Applica la condizione elemento per elemento: se finito e > 0 tieni il valore, altrimenti 0
            err = np.where(np.isfinite(phot_err) & (phot_err > 0), phot_err, 0.0)
        else:
            err = np.zeros(n)

        ax.errorbar(
            x,
            phot_flux,
            yerr=err,
            fmt="o",
            color=C_PHOT_FLUX,
            ecolor=C_PHOT_ERR,
            elinewidth=1.4,
            capsize=3,
            capthick=1.4,
            markersize=6,
            markeredgewidth=0,
            zorder=3,
            label="phot flux",
        )
        ax.plot(x, phot_flux, color=C_PHOT_FLUX, linewidth=0.7, alpha=0.4, zorder=2)

        ax.set_xlim(-0.6, n - 0.4)
        ax.set_ylim(_robust_ylim(phot_flux))

        ax.set_xlabel("Filter")
        ax.set_ylabel("Flux [maggies]")
        ax.set_title(_title_from_data(data, version), pad=10)

        if filter_labels:
            ax.set_xticks(x)
            ax.set_xticklabels(filter_labels, rotation=45, ha="right", fontsize=7.5)
        else:
            ax.set_xticks(x)

        ax.legend(
            framealpha=0.15, edgecolor="#2e3650", labelcolor="#c8d0e8", fontsize=8
        )

        if standalone:
            fig.tight_layout()
            if show:
                plt.show()

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 — PANNELLO COMPLETO (spettro + fotometria + S/N)
# ─────────────────────────────────────────────────────────────────────────────


def plot_full(
    filepath: str,
    version: str = "V1",
    show: bool = True,
    figsize: tuple = (15, 9),
    rest_frame: bool = False,
) -> plt.Figure:
    """
    Pannello a 3 righe:
      [0] Spettro flux ± err con bande mascherate
      [1] S/N ratio con linea a S/N=3
      [2] SED fotometrica
    """
    data = _load_version(filepath, version)

    wave = data.get("wave")
    flux = data.get("flux")
    flux_err = data.get("flux_err")
    mask = data.get("mask")
    z = data.get("redshift", 0.0)

    phot_flux = data.get("phot_flux")
    phot_err = data.get("phot_err")
    filters = data.get("phot_filters")

    has_spec = wave is not None and flux is not None
    has_phot = phot_flux is not None
    has_sn = has_spec and flux_err is not None

    # Calcola altezze righe dinamicamente
    heights = []
    if has_spec:
        heights.append(3)
    if has_sn:
        heights.append(1.2)
    if has_phot:
        heights.append(2)
    n_rows = len(heights)

    if n_rows == 0:
        raise ValueError("Nessun dato da plottare.")

    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(
            n_rows,
            1,
            figsize=figsize,
            gridspec_kw={"height_ratios": heights, "hspace": 0.07},
        )
        if n_rows == 1:
            axes = [axes]

        ax_iter = iter(axes)

        wave_plot = wave / (1.0 + float(z)) if rest_frame else wave
        xlabel = (
            r"$\lambda_\mathrm{rest}$ [µm]"
            if rest_frame
            else r"$\lambda_\mathrm{obs}$ [µm]"
        )
        xlim = _robust_xlim(wave_plot) if has_spec else None

        # ── ROW 0: Spettro ───────────────────────────────────────────────────
        if has_spec:
            ax0 = next(ax_iter)

            if flux_err is not None:
                ax0.fill_between(
                    wave_plot,
                    flux - flux_err,
                    flux + flux_err,
                    color=C_ERR,
                    alpha=0.35,
                    linewidth=0,
                    zorder=2,
                )
            ax0.plot(
                wave_plot,
                flux,
                color=C_FLUX,
                linewidth=0.9,
                alpha=0.92,
                zorder=3,
                label="flux",
            )

            _draw_mask_bands(ax0, wave_plot, mask)

            ax0.set_xlim(xlim)
            ax0.set_ylim(_robust_ylim(flux, mask))
            ax0.set_ylabel("Flux [maggies]")
            ax0.set_title(_title_from_data(data, version), pad=10, fontsize=13)
            ax0.tick_params(labelbottom=False)
            ax0.legend(
                framealpha=0.15,
                edgecolor="#2e3650",
                labelcolor="#c8d0e8",
                fontsize=8,
                loc="upper right",
            )

            if mask is not None:
                n_good = mask.sum()
                n_total = len(mask)
                ax0.text(
                    0.01,
                    0.96,
                    f"coverage  {n_good}/{n_total}  ({100 * n_good / n_total:.1f}%)",
                    transform=ax0.transAxes,
                    va="top",
                    ha="left",
                    fontsize=8,
                    color="#7a8ab0",
                )

        # ── ROW 1: S/N ──────────────────────────────────────────────────────
        if has_sn:
            ax1 = next(ax_iter)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                sn = np.where(flux_err > 0, flux / flux_err, np.nan)

            ax1.plot(
                wave_plot, sn, color="#a3c9ff", linewidth=0.75, alpha=0.8, zorder=3
            )
            ax1.axhline(
                3.0,
                color=C_ZLINE,
                linewidth=0.8,
                linestyle="--",
                alpha=0.7,
                zorder=2,
                label="S/N = 3",
            )
            ax1.axhline(0.0, color="#2e3650", linewidth=0.6, zorder=1)

            _draw_mask_bands(ax1, wave_plot, mask)

            ax1.set_xlim(xlim)
            ax1.set_ylim(_robust_ylim(sn, mask, lo_pct=0.5, hi_pct=99.5))
            ax1.set_ylabel("S / N", fontsize=9)
            ax1.tick_params(labelbottom=False)
            ax1.legend(
                framealpha=0.15,
                edgecolor="#2e3650",
                labelcolor="#c8d0e8",
                fontsize=7.5,
                loc="upper right",
            )

        # ── ROW 2: Fotometria ────────────────────────────────────────────────
        if has_phot:
            ax2 = next(ax_iter)
            n = len(phot_flux)
            x = np.arange(n)
            
            if phot_err is not None:
                # Applica la condizione elemento per elemento: se finito e > 0 tieni il valore, altrimenti 0
                err = np.where(np.isfinite(phot_err) & (phot_err > 0), phot_err, 0.0)
            else:
                err = np.zeros(n)

            ax2.errorbar(
                x,
                phot_flux,
                yerr=err,
                fmt="o",
                color=C_PHOT_FLUX,
                ecolor=C_PHOT_ERR,
                elinewidth=1.4,
                capsize=3,
                capthick=1.4,
                markersize=6,
                markeredgewidth=0,
                zorder=3,
                label="photometry",
            )
            ax2.plot(
                x, phot_flux, color=C_PHOT_FLUX, linewidth=0.7, alpha=0.4, zorder=2
            )

            ax2.set_xlim(-0.6, n - 0.4)
            ax2.set_ylim(_robust_ylim(phot_flux))
            ax2.set_ylabel("Flux [maggies]", fontsize=9)
            ax2.set_xlabel("Filter index")
            ax2.legend(
                framealpha=0.15, edgecolor="#2e3650", labelcolor="#c8d0e8", fontsize=8
            )

            if filters is not None:
                lbls = [
                    f.decode() if isinstance(f, (bytes, np.bytes_)) else str(f)
                    for f in filters
                ]
                ax2.set_xticks(x)
                ax2.set_xticklabels(lbls, rotation=45, ha="right", fontsize=7.5)

        # Ultima riga: xlabel spettrale
        if has_spec and (has_sn or not has_phot):
            axes[-1].set_xlabel(xlabel)
        elif has_spec:
            pass  # la fotometria ha il suo xlabel

        fig.tight_layout()
        if show:
            plt.show()

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 4 — CONFRONTO DUE VERSIONI (es. V1 vs cluster)
# ─────────────────────────────────────────────────────────────────────────────


def plot_compare_versions(
    filepath: str,
    version_a: str = "V1",
    version_b: str = "cluster",
    show: bool = True,
    figsize: tuple = (15, 6),
    rest_frame: bool = False,
) -> plt.Figure:
    """
    Confronta due versioni dello stesso file affiancando spettro e maschera.

    Pannello superiore: entrambi i flussi sovrapposti.
    Pannello inferiore: differenza delle maschere (pixel aggiunti/rimossi).
    """
    da = _load_version(filepath, version_a)
    db = _load_version(filepath, version_b)

    wave = da.get("wave")
    flux = da.get("flux")
    err = da.get("flux_err")
    mask_a = da.get("mask")
    mask_b = db.get("mask")
    z = da.get("redshift", 0.0)

    if wave is None or flux is None:
        raise ValueError("Spectroscopy mancante.")

    wave_plot = wave / (1.0 + float(z)) if rest_frame else wave
    xlabel = (
        r"$\lambda_\mathrm{rest}$ [µm]"
        if rest_frame
        else r"$\lambda_\mathrm{obs}$ [µm]"
    )

    C_B = "#f5a05a"  # colore per versione B

    with plt.rc_context(STYLE):
        fig, (ax0, ax1) = plt.subplots(
            2, 1, figsize=figsize, gridspec_kw={"height_ratios": [3, 1], "hspace": 0.07}
        )

        xlim = _robust_xlim(wave_plot)

        # ── Pannello superiore: spettro ──────────────────────────────────────
        if err is not None:
            ax0.fill_between(
                wave_plot,
                flux - err,
                flux + err,
                color=C_ERR,
                alpha=0.25,
                linewidth=0,
                zorder=2,
            )
        ax0.plot(
            wave_plot,
            flux,
            color=C_FLUX,
            linewidth=0.85,
            zorder=3,
            label=f"flux  [{version_a}]",
        )

        # Maschera A (grigio scuro)
        _draw_mask_bands(ax0, wave_plot, mask_a)

        # Maschera B (arancio) — solo regioni DIVERSE
        if mask_b is not None and mask_a is not None:
            diff_removed = mask_a & ~mask_b  # erano buoni, ora mascherati
            diff_added = ~mask_a & mask_b  # erano mascherati, ora buoni
            # Evidenzia in rosso i pixel rimossi da B
            for arr, col, lbl in [
                (diff_removed, "#e05050", f"removed in {version_b}"),
                (diff_added, "#50c87a", f"added in {version_b}"),
            ]:
                if arr.any():
                    diffs = np.diff(arr.astype(int), prepend=0, append=0)
                    ss = np.where(diffs == 1)[0]
                    es = np.where(diffs == -1)[0]
                    hw = np.median(np.diff(wave)) / 2
                    for s, e in zip(ss, es):
                        ax0.axvspan(
                            wave_plot[s] - hw,
                            wave_plot[e - 1] + hw,
                            color=col,
                            alpha=0.25,
                            linewidth=0,
                            zorder=1,
                        )
                    ax0.axvspan(0, 0, color=col, alpha=0.5, linewidth=0, label=lbl)

        ax0.set_xlim(xlim)
        ax0.set_ylim(_robust_ylim(flux, mask_a))
        ax0.set_ylabel("Flux [maggies]")
        ax0.set_title(_title_from_data(da, f"{version_a}  vs  {version_b}"), pad=10)
        ax0.tick_params(labelbottom=False)
        ax0.legend(
            framealpha=0.15,
            edgecolor="#2e3650",
            labelcolor="#c8d0e8",
            fontsize=8,
            loc="upper right",
        )

        # ── Pannello inferiore: delta maschera ───────────────────────────────
        if mask_a is not None and mask_b is not None:
            delta = mask_b.astype(int) - mask_a.astype(int)  # +1, 0, -1
            colors = np.where(
                delta > 0, "#50c87a", np.where(delta < 0, "#e05050", "#2e3650")
            )

            ax1.bar(
                wave_plot,
                delta,
                width=np.median(np.diff(wave)),
                color=colors,
                alpha=0.75,
                zorder=3,
            )
            ax1.axhline(0, color="#2e3650", linewidth=0.6, zorder=1)
            ax1.set_ylim(-1.5, 1.5)
            ax1.set_yticks([-1, 0, 1])
            ax1.set_yticklabels(["removed", "same", "added"], fontsize=8)
            ax1.set_ylabel("Δ mask", fontsize=9)
        else:
            ax1.text(
                0.5,
                0.5,
                "mask non disponibile",
                transform=ax1.transAxes,
                ha="center",
                va="center",
                color="#7a8ab0",
            )

        ax1.set_xlim(xlim)
        ax1.set_xlabel(xlabel)

        fig.tight_layout()
        if show:
            plt.show()

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# QUICK INSPECT — stampa la struttura del file
# ─────────────────────────────────────────────────────────────────────────────


def list_versions(filepath: str) -> list[str]:
    """Restituisce la lista dei gruppi top-level nel file H5."""
    with h5py.File(filepath, "r") as f:
        versions = list(f.keys())
    print(f"  {filepath}")
    print(f"  Versioni disponibili: {versions}")
    return versions

# # ─────────────────────────────────────────────────────────────────────────────
# # ESEMPIO D'USO
# # ─────────────────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     import os  # Puoi anche spostarlo in cima al file con gli altri import

#     FILE = "/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/capers-egs61-v4_prism-clear_6368_24713.h5"
#     OUT_DIR = "/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots"

#     # 1. Crea la cartella di destinazione se non esiste già
#     os.makedirs(OUT_DIR, exist_ok=True)

#     # 2. Estrai il nome base del file H5 (senza il percorso e senza l'estensione .h5)
#     base_name = os.path.splitext(os.path.basename(FILE))[0]

#     # Versioni disponibili
#     list_versions(FILE)

#     print("\nSalvataggio dei plot in corso...")

#     # --- Solo spettro ---
#     fig_spec = plot_spectrum(FILE, version="V1", show=False)
#     out_spec = os.path.join(OUT_DIR, f"{base_name}_spectrum.png")
#     # fig_spec.savefig(out_spec, dpi=200, bbox_inches="tight")
#     print(f"  ✓ Salvato: {out_spec}")

#     # --- Solo fotometria ---
#     fig_phot = plot_photometry(FILE, version="V1", show=False)
#     out_phot = os.path.join(OUT_DIR, f"{base_name}_photometry.png")
#     # fig_phot.savefig(out_phot, dpi=200, bbox_inches="tight")
#     print(f"  ✓ Salvato: {out_phot}")

#     # --- Pannello completo ---
#     fig_full = plot_full(FILE, version="V1", rest_frame=False, show=False)
#     out_full = os.path.join(OUT_DIR, f"{base_name}_full.png")
#     fig_full.savefig(out_full, dpi=200, bbox_inches="tight")
#     print(f"  ✓ Salvato: {out_full}")

#     # --- Confronto V1 vs cluster ---
#     fig_comp = plot_compare_versions(
#         FILE, version_a="V1", version_b="cluster", show=False
#     )
#     out_comp = os.path.join(OUT_DIR, f"{base_name}_compare.png")
#     fig_comp.savefig(out_comp, dpi=200, bbox_inches="tight")
#     print(f"  ✓ Salvato: {out_comp}")

#     # (Opzionale) Decommenta la riga qui sotto se, oltre a salvarli, vuoi anche vederli a schermo alla fine
#     # plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ESEMPIO D'USO
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    for FILE in sorted(glob.glob("/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/*.h5")):
        
        OUT_DIR = "/Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots"
        os.makedirs(OUT_DIR, exist_ok=True)

        # 2. Estrai il nome base del file H5 (senza il percorso e senza l'estensione .h5)
        base_name = os.path.splitext(os.path.basename(FILE))[0]

        # Versioni disponibili
        list_versions(FILE)

        print("\nSalvataggio dei plot in corso...")

        # --- Solo spettro ---
        fig_spec = plot_spectrum(FILE, version="V1", show=False)
        out_spec = os.path.join(OUT_DIR, f"{base_name}_spectrum.png")
        # fig_spec.savefig(out_spec, dpi=200, bbox_inches="tight")
        print(f"  ✓ Salvato: {out_spec}")

        # --- Solo fotometria ---
        fig_phot = plot_photometry(FILE, version="V1", show=False)
        out_phot = os.path.join(OUT_DIR, f"{base_name}_photometry.png")
        # fig_phot.savefig(out_phot, dpi=200, bbox_inches="tight")
        print(f"  ✓ Salvato: {out_phot}")

        # --- Pannello completo ---
        fig_full = plot_full(FILE, version="cluster", rest_frame=False, show=False)
        out_full = os.path.join(OUT_DIR, f"{base_name}_full.png")
        fig_full.savefig(out_full, dpi=200, bbox_inches="tight")
        print(f"  ✓ Salvato: {out_full}")

        # --- Confronto V1 vs cluster ---
        fig_comp = plot_compare_versions(
            FILE, version_a="V1", version_b="cluster", show=False
        )
        out_comp = os.path.join(OUT_DIR, f"{base_name}_compare.png")
        # fig_comp.savefig(out_comp, dpi=200, bbox_inches="tight")
        print(f"  ✓ Salvato: {out_comp}")


  /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/capers-egs61-v4_prism-clear_6368_24713.h5
  Versioni disponibili: ['V1', 'cluster']

Salvataggio dei plot in corso...
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/capers-egs61-v4_prism-clear_6368_24713_spectrum.png
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/capers-egs61-v4_prism-clear_6368_24713_photometry.png


/var/folders/mw/d8_gfg4s5tq10rtwbkqwzxw00000gn/T/ipykernel_6770/2350633820.py:598: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/capers-egs61-v4_prism-clear_6368_24713_full.png


/var/folders/mw/d8_gfg4s5tq10rtwbkqwzxw00000gn/T/ipykernel_6770/2350633820.py:748: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/capers-egs61-v4_prism-clear_6368_24713_compare.png
  /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/capers-udsp1-v4_prism-clear_6368_21205.h5
  Versioni disponibili: ['V1', 'V2', 'cluster']

Salvataggio dei plot in corso...
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/capers-udsp1-v4_prism-clear_6368_21205_spectrum.png
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/capers-udsp1-v4_prism-clear_6368_21205_photometry.png


...

/var/folders/mw/d8_gfg4s5tq10rtwbkqwzxw00000gn/T/ipykernel_6770/2350633820.py:748: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/uncover-v4_prism-clear_2561_18407_compare.png
  /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/h5/uncover-v4_prism-clear_2561_23955.h5
  Versioni disponibili: ['V1', 'cluster']

Salvataggio dei plot in corso...
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/uncover-v4_prism-clear_2561_23955_spectrum.png
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/uncover-v4_prism-clear_2561_23955_photometry.png


/var/folders/mw/d8_gfg4s5tq10rtwbkqwzxw00000gn/T/ipykernel_6770/2350633820.py:598: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/uncover-v4_prism-clear_2561_23955_full.png
  ✓ Salvato: /Users/matteo/Dottorato/DATA/DAWN_Archive/prospector/golden_sample/plots/uncover-v4_prism-clear_2561_23955_compare.png


/var/folders/mw/d8_gfg4s5tq10rtwbkqwzxw00000gn/T/ipykernel_6770/2350633820.py:748: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
